# పాఠం 11 - ఏజెంట్-టు-ఏజెంట్ (A2A) ప్రోటోకాల్


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## A2A ప్రోటోకాల్ ఏమిటి?

**ఏజెంట్-టు-ఏజెంట్ (A2A) ప్రోటోకాల్** అనేది ఒక ఓపెన్ స్టాండర్డ్, ఇది AI ఏజెంట్లు పరస్పరం కమ్యూనికేట్ చేయడానికి,
ఒకరినొకరు కనుగొనడం మరియు కలిసి పనిచేయడానికి అనుమతిస్తుంది — అవి వేరే వేరే ఫ్రేమ్‌వర్క్‌లపై నిర్మించబడ్డా
లేదా వేరే వేరే సేవల ద్వారా నిర్వహింపబడాడా సరే.

ముఖ్యమైన భావనలు:

- **అన్వేషణ** – ఏజెంట్లు తమ సామర్థ్యాలను వివరించే *ఏజెంట్ కార్డ్*ని ప్రచురిస్తారు, దీని ద్వారా
  ఇతర ఏజెంట్లు (లేదా ఆర్కెస్ట్‌రేటర్లు) సరైన నిపుణుడిని ఒక పనికి సులభంగా కనుగొనవచ్చు.
- **మెసేజ్ పంపిణీ** – ఏజెంట్లు సాధారణ ప్రోటోకాల్ ద్వారా నిర్మిత సందేశాలను మార్చుకుంటాయి, అందువల్ల ఒక
  ఏజెంట్ నుండి వచ్చిన అభ్యర్థన అంతర్గత అమలు విధానం ఏదైనా సరే, ఇంకొక ఏజెంట్ అర్థం చేసుకుని నిర్వర్తించగలదు.

- **పని జీవన చక్రం** – A2A *సబ్మిట్ చేసిన*, *పని చేస్తోంది*, *పూర్తయింది*, మరియు
  *విఫలమైంది* వంటి స్థితులను నిర్వచిస్తుంది, దీని ద్వారా ఆర్కెస్ట్‌రేటర్ అప్పగించిన పని ఎలా సాగుతోందో పూర్తిగా కనిపిస్తుంది.

ఈ పాఠంలో మేము మూడు ప్రత్యేకకరణల ట్రావెల్ ఏజెంట్లను ఒక వర్క్‌ఫ్లోలో కనెక్ట్ చేసి,
ప్రతి ఏజెంట్ తన నైపుణ్యాన్ని అందించి తదుపరి వ్య‌క్తికి ఫలితాలను పంపడం ద్వారా A2A శైలిలో సహకారాన్ని అనుకరించడం జరుగుతుంది.


## ప్రత్యేక ప్రయాణ ఏజెంట్లను సృష్టించడం


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## వర్క్‌ఫ్లో ద్వారా బహుళ-ఏజెంట్ సహకారం

మేము మూడు ఏజెంట్లను A2A మెస్సేజ్ పాసింగ్‌ను ప్రతిబింబించే వరస వర్క్‌ఫ్లోలో కలిపి ఉంటాము:

1. **CurrencyExchangeAgent** యూజర్ అభ్యర్థనను స్వీకరిస్తుంది మరియు కరెన్సీ మార్గదర్శకత్వాన్ని ఉత్పత్తి చేస్తుంది.
2. **ActivityPlannerAgent** సం‍పూర్ణ పరిస్తితిని స్వీకరిస్తుంది మరియు కార్యాచరణ సిఫారసులను జతచేస్తుంది.
3. **TravelManagerAgent** రెండు ఇన్‌పుట్‌లను కూడా సమ్మిళితం చేసి తుదిగంట ప్రయాణ సారాంశాన్ని తయారు చేస్తుంది.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ఉత్పత్తిలో A2Aని అర్థం చేసుకోవటం

ఉత్పత్తి వాతావరణంలో A2A ప్రోటోకాల్ శక్తివంతమైన క్రాస్-సర్వీస్ సన్నివేశాలను అనుమతిస్తుంది:

| సామర్థ్యం | వివరణ |
|---|---|
| **క్రాస్-ఫ్రేమ్‌వర్క్ అంతరచేతనం** | ఒక ఫ్రేమ్‌వర్క్‌తో నిర్మించబడిన ఏజెంట్ ఏ ఇతర A2A-అనుకూల ఫ్రేమ్‌వర్క్‌తో నిర్మించబడిన ఏజెంట్‌కు పనులను అప్పగించగలదు, నిజమైన క్రాస్-సంస్థ అంతరచేతనం సాధ్యమవుతుంది. |
| **సర్వీస్ సరిహద్దులు** | ఏజెంట్లు వేరైన మైక్రోసర్వీసుల్లో, క్లౌడ్ ప్రాంతాలలో లేదా భిన్న సంస్థలలో ఉండవచ్చు కానీ సజావుగా సహకరించగలవు. |
| **డయనమిక్ కనుగొనటం** | ఒక ఆర్కెస్ట్రేటర్ రన్‌టైమ్‌లో ఏజెంట్ కార్డ్ రిజిస్ట్రీని ప్రశ్నించి నిర్దిష్ట ఉప-పనికి అత్యుత్తమ నిపుణుడిని కనుగొనవచ్చు. |
| **స్ట్రీమింగ్ & పుష్ నోటిఫికేషన్లు** | A2A నిజకాల పురోగతి నవీకరణలకు సర్వర్-సెంట్ ఈవెంట్స్ (SSE) మరియు దీర్ఘకాలిక పనుల కోసం పుష్ నోటిఫికేషన్లను మద్దతు ఇస్తుంది. |

మేము పైగా రూపొందించిన వర్క్‌ఫ్లో ఈ నమూనా యొక్క సరళీకృత, ఇన్-ప్రొసెస్ వెర్షన్. నిజమైన
డిప్లాయ్‌మెంట్‌లో ప్రతి ఏజెంట్ HTTP ఎండ్పాయింట్‌ను బహిరంగం చేస్తుంది, ఏజెంట్ కార్డ్‌ను ప్రచురిస్తుంది, మరియు
A2A JSON-RPC ప్రోటోకాల్ ద్వారా కమ్యూనికేట్ చేయును.


## సారాంశం

ఈ పాఠంలో మీరు నేర్చుకున్నది:

1. **A2A ప్రోటోకాల్ ఏమిటి** — ఏజెంట్-టు-ఏజెంట్ డిస్కవరీ, సందేశాలు మరియు
   పని నిర్వహణ కోసం ఓపెన్ స్టాండర్డ్.
2. **ప్రత్యేక ఏజెంట్లను ఎలా సృష్టించాలి** — కరెన్సీ ఎక్స్కేంజ్ ఏజెంట్, యాక్టివిటీ ప్లానర్ ఏజెంట్,
   మరియు ట్రావెల్ మేనేజర్ ఆర్కెస్ట్రేటర్.
3. **ఏజెంట్లను వర్క్ֆ్లోలో ఎలా కనెక్ట్ చేయాలి** — ఏజెంట్ల మధ్య క్ర‌మా‌నుగుణ సందేశ పంప‌కాన్ని
   మోడల్ చేయడానికి `WorkflowBuilder` ఉపయోగించడం.
4. **ఉత్పాద‌నంలో A2A ఎలా పనిచేస్తుంది** — డైనమిక్ డిస్కవరీ మరియు స్ట్రీమింగ్ అప్‌డేట్ల
   ద్వార cross-framework, cross-service సహకారాన్ని సాధ్యం చేస్తుంది.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
